In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [3]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

In [4]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")

sentiment_model = model.with_structured_output(SentimentSchema)


In [5]:
prompt = f"I really love this product! It's amazing!"
response = sentiment_model.invoke(prompt)
print(response)





sentiment='positive'


In [6]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [7]:
def find_sentiment(state: ReviewState) :
    prompt = f"Find the sentiment of the following review: {state['review']}"
    response = sentiment_model.invoke(prompt)
    return {'sentiment': response.sentiment}

In [8]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)

graph.add_edge(START, 'find_sentiment')
graph.add_edge('find_sentiment', END)

workflow = graph.compile()

In [9]:
initial_state = {"review": "This is the best ice cream I have ever had."}

final_result = workflow.invoke(initial_state)
print(final_result)

{'review': 'This is the best ice cream I have ever had.', 'sentiment': 'positive'}
